In [ ]:
import time
global_end_time = time.time() + 12 * 3600 - 600

In [ ]:
# Cell 00 — F1 setup (paste as the FIRST code cell, or merge into the notebook's existing first cell).
# Keep the live notebook's own global_end_time if it already defines one at the very top; this only fills it in.
import os, time
os.environ["ARC_SHARD"] = "0"          # SMOKE: 60-task eval shard (ignored in competition rerun). Remove for full run.
os.environ["PYTHONHASHSEED"] = "0"
try: global_end_time
except NameError: global_end_time = time.time() + 12 * 3600 - 600
RERUN = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
COMP = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"        # same root starter.py uses
TEST_CHALLENGES_PATH = f"{COMP}/arc-agi_test_challenges.json" if RERUN else f"{COMP}/arc-agi_evaluation_challenges.json"
os.makedirs("/kaggle/working/selector", exist_ok=True); open("/kaggle/working/selector/__init__.py", "a").close()
print("rerun:", RERUN, "| test:", TEST_CHALLENGES_PATH, "| hours left:", round((global_end_time - time.time()) / 3600, 2))


In [ ]:
!pip uninstall -y tensorflow

In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score


In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")


In [ ]:
%%writefile arc_solver_base.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch



def _resolve_model_dir():
    """Find the attached qwen3 model dir under /kaggle/input (layout changed across Kaggle images)."""
    import os
    from pathlib import Path
    candidates = [
        "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/models/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
    ]
    for c in candidates:
        if Path(c, "config.json").is_file():
            return c
    hits = []
    root = Path("/kaggle/input")
    if root.is_dir():
        for p in root.rglob("config.json"):
            s = str(p.parent)
            if "qwen3_4b" in s.lower() or "qwen3-4b" in s.lower() or "grids15" in s.lower():
                hits.append(str(p.parent))
    if len(hits) == 1:
        return hits[0]
    if hits:
        ranked = sorted(hits, key=lambda h: (0 if "bfloat16" in h else 1, len(h)))
        return ranked[0]
    top = sorted(os.listdir("/kaggle/input")) if root.is_dir() else []
    raise FileNotFoundError(
        f"No qwen3 model config.json under /kaggle/input. top-level={top} candidates_tried={candidates}"
    )

def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    
    MODEL_DIR = _resolve_model_dir()
    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_DIR,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = os.environ.get("ARC_OUT_DIR", "/kaggle/inference_outputs")  # F1 edit 1
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        if time.time() > end_time - float(os.environ.get("ARC_START_GUARD_S", "1200")):  # F1 edit 2
            print(f"[Rank {rank}] skip {key}: not enough time left to finish")
            continue
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile arc_solver_views4.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch



def _resolve_model_dir():
    """Find the attached qwen3 model dir under /kaggle/input (layout changed across Kaggle images)."""
    import os
    from pathlib import Path
    candidates = [
        "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/models/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
    ]
    for c in candidates:
        if Path(c, "config.json").is_file():
            return c
    hits = []
    root = Path("/kaggle/input")
    if root.is_dir():
        for p in root.rglob("config.json"):
            s = str(p.parent)
            if "qwen3_4b" in s.lower() or "qwen3-4b" in s.lower() or "grids15" in s.lower():
                hits.append(str(p.parent))
    if len(hits) == 1:
        return hits[0]
    if hits:
        ranked = sorted(hits, key=lambda h: (0 if "bfloat16" in h else 1, len(h)))
        return ranked[0]
    top = sorted(os.listdir("/kaggle/input")) if root.is_dir() else []
    raise FileNotFoundError(
        f"No qwen3 model config.json under /kaggle/input. top-level={top} candidates_tried={candidates}"
    )

def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.float32).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    
    MODEL_DIR = _resolve_model_dir()
    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_DIR,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = os.environ.get("ARC_OUT_DIR", "/kaggle/inference_outputs")  # F1 edit 1
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        if time.time() > end_time - float(os.environ.get("ARC_START_GUARD_S", "1200")):  # F1 edit 2
            print(f"[Rank {rank}] skip {key}: not enough time left to finish")
            continue
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=4, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        def _geo(sk):  # (transposed?, odd rot90 count?) -> same grid shapes within a group
            p = sk.split(".")[1:]
            return ("transpose" in p, p.count("rot90") % 2)
        batches = []
        for tp in (False, True):
            for test_id, subkeys in test_id_to_subkeys.items():
                for par in (0, 1):
                    grp = [s for s in subkeys if _geo(s) == (tp, par)]
                    batches.extend(grp[i:i+4] for i in range(0, len(grp), 4))

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                if len(set(map(len, tokens))) == 1:
                    dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)
                else:  # safety fallback: decode one at a time
                    print(f"[Rank {rank}] unequal lengths {[len(t) for t in tokens]}, decoding singly")
                    dfs_result = [(i, r) for i, t in enumerate(tokens) for _, r in inference_turbo_dfs(model, [t], max_new_tokens, max_score, end_time)]

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")


In [ ]:
%%writefile starter.py
import os
import sys
# Demote Kaggle utility that ships a broken torch/ so site-packages wins (F1 smoke v1 ImportError).
for _p in list(sys.path):
    if "pip_install_unsloth_flash_patch" in _p:
        try:
            sys.path.remove(_p)
            sys.path.append(_p)
        except ValueError:
            pass
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    order_file = os.environ.get("ARC_TASK_ORDER")  # F1: pass B queue, least-confident first
    if order_file and os.path.exists(order_file):
        keys = [k for k in json.load(open(order_file))["order"] if k in data]
    else:
        keys = sorted(data.keys())
        if not rerun_mode and os.environ.get("ARC_SHARD"):  # offline sharding only
            keys = keys[int(os.environ["ARC_SHARD"])::2]
    for key in keys:
        queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)


In [ ]:
%%writefile /kaggle/working/selector/arch_ingest.py
"""Ingest ARChitects-style decoder outputs (bz2 pickles in /kaggle/inference_outputs) into our schema.

Each pickle holds candidates for one subkey; each candidate is {beam_score, score_aug, solution}
(beam_score, score_aug entries are NLLs -> lower is better). We do not have the exact pickle container
from the notebook, so the loader accepts: a dict {subkey: [cand,...]}, a dict {subkey: cand}, a list of cands,
or a single cand. Subkey -> (task_id, test_idx) via '<task>_<idx>' (extra suffixes after the idx are ignored).

ALIGNMENT: score_aug is a positional list. Our per-view weights assume position j is the same view for every
candidate. check_alignment() flags groups whose score_aug lengths differ; it cannot detect a reordering, so
verify in arc_solver.calc_scores that the aug list is built from a fixed, seeded order (not resampled per candidate).
"""
import bz2, glob, hashlib, json, os, pickle, re
from collections import defaultdict
import numpy as np

def _cands(obj, default_key):
    if isinstance(obj, dict) and "solution" in obj: yield default_key, obj; return
    if isinstance(obj, dict):
        for k, v in obj.items():
            for kk, c in _cands(v, str(k)): yield kk, c
        return
    if isinstance(obj, (list, tuple)):
        for v in obj:
            for kk, c in _cands(v, default_key): yield kk, c

def parse_subkey(sk):
    m = re.match(r"^([0-9a-f]{8})(?:_(\d+))?", os.path.basename(str(sk)))
    return (m.group(1), int(m.group(2) or 0)) if m else (str(sk), 0)

def _grid(sol):
    a = np.asarray(sol); return a.astype(int).tolist()

def _h(grid): return hashlib.md5(json.dumps(grid).encode()).hexdigest()

def load_pickles(root):
    """-> {(task_id, test_idx): {hash: {"grid", "guesses": [cand,...]}}}"""
    out = defaultdict(dict)
    files = sorted(glob.glob(os.path.join(root, "**", "*"), recursive=True))
    for f in files:
        if os.path.isdir(f): continue
        try:
            with (bz2.open(f, "rb") if f.endswith(".bz2") or open(f, "rb").read(3) == b"BZh" else open(f, "rb")) as fh:
                obj = pickle.load(fh)
        except Exception:
            continue
        rel = os.path.relpath(f, root).split(os.sep)
        default = rel[0] if len(rel) > 1 else os.path.splitext(os.path.splitext(rel[0])[0])[0]
        for sk, c in _cands(obj, default):
            key = parse_subkey(sk); g = _grid(c["solution"]); h = _h(g)
            out[key].setdefault(h, {"grid": g, "guesses": []})["guesses"].append(c)
    return dict(out)

def check_alignment(groups):
    """Returns list of (key, sorted set of score_aug lengths) where lengths differ."""
    bad = []
    for key, sols in groups.items():
        L = {len(c.get("score_aug", [])) for s in sols.values() for c in s["guesses"]}
        if len(L) > 1: bad.append((key, sorted(L)))
    return bad

def to_rows(groups, n_views=None):
    """Our schema: logprobs[view] = -mean NLL at aug position j over guesses of this solution;
    plus 'beam' view = -min beam_score, and meta fields used by the A0 baselines."""
    rows = []
    for (tid, ti), sols in groups.items():
        for h, s in sols.items():
            G = s["guesses"]; aug = [np.asarray(c.get("score_aug", []), float) for c in G]
            L = n_views or max((len(a) for a in aug), default=0)
            lp = {}
            for j in range(L):
                vals = [a[j] for a in aug if len(a) > j]
                if vals: lp[f"aug{j}"] = -float(np.mean(vals))
            bs = [float(c["beam_score"]) for c in G if c.get("beam_score") is not None]
            if bs: lp["beam"] = -float(min(bs))
            lp["count"] = float(len(G))  # number of sampled hits (the kgmon "inf_score" term)
            rows.append({"task_id": tid, "test_idx": ti, "grid": s["grid"], "logprobs": lp,
                         "n_samples": len(G),
                         "_beam_scores": bs, "_score_aug": [a.tolist() for a in aug]})
    return rows

# --- A0 / A0' baselines: reproduce notebook getters (higher = better) ---
def getter_kgmon(guesses):
    """len(guesses) - mean over guesses of mean(score_aug)."""
    return len(guesses) - float(np.mean([np.mean(g["score_aug"]) for g in guesses]))

def getter_full_probmul_3(guesses):
    """sum over guesses of (3 - beam_score) + mean over guesses of sum(3 - score_aug)."""
    a = sum(3 - float(g["beam_score"]) for g in guesses)
    b = float(np.mean([np.sum(3 - np.asarray(g["score_aug"], float)) for g in guesses]))
    return a + b

def baseline_submission(groups, tasks, getter=getter_kgmon):
    sub = {}
    for tid, t in tasks.items():
        sub[tid] = []
        for ti, tt in enumerate(t["test"]):
            sols = list(groups.get((tid, ti), {}).values())
            sols.sort(key=lambda s: -getter(s["guesses"]))
            gs = [s["grid"] for s in sols[:2]] or [tt["input"]]
            while len(gs) < 2: gs.append(gs[0])
            sub[tid].append({"attempt_1": gs[0], "attempt_2": gs[1]})
    return sub


In [ ]:
%%writefile /kaggle/working/selector/calibrate.py
"""Fit per-view weights a_v (= w_v/T_v; w and T are not separately identifiable for a linear score)
and verifier weight w_ver by minimising softmax log-loss over candidates within each (task,test_idx) group.
s(c) = sum_v a_v * logp_v(c) + w_ver * ver(c). Plus ECE of the calibrated top-1 probability."""
import json, numpy as np

def features(group, views, ver=None):
    X = np.array([[c["logprobs"].get(v, np.nan) for v in views] for c in group], float)
    for j in range(X.shape[1]):  # missing view -> group min for that view
        col = X[:, j]; mn = np.nanmin(col) if np.isfinite(col).any() else -50.0
        col[np.isnan(col)] = mn; X[:, j] = col
    X = X - X.max(0, keepdims=True)  # per-view shift; softmax invariant, improves conditioning
    v = np.array([ver[i] if ver is not None else 0.0 for i in range(len(group))])[:, None]
    return np.hstack([X, v])

def _softmax(z): z = z - z.max(); e = np.exp(z); return e / e.sum()

def fit(data, n_views, l2=1e-3, lr=0.05, iters=500, init=None, ver_nonneg=True):
    """L-BFGS-B (scipy) with bounds: view weights >= 0, verifier weight >= 0 (an expert can only add evidence).
    Falls back to projected gradient descent if scipy is unavailable."""
    data = [(X, y) for X, y in data if y.sum() > 0 and len(y) > 1]
    x0 = np.array(init, float) if init is not None else np.r_[np.full(n_views, 1.0 / max(n_views, 1)), 0.0]
    try:
        from scipy.optimize import minimize
        def f(th):
            L = 0.5 * l2 * th @ th; g = l2 * th
            for X, y in data:
                z = X @ th; z = z - z.max(); e = np.exp(z); p = e / e.sum(); t = y / y.sum()
                L -= np.log(max(p[y > 0].sum(), 1e-300)) / len(data); g += X.T @ (p - t) / len(data)
            return L, g
        b = [(0, None)] * n_views + [((0, None) if ver_nonneg else (None, None))]
        return minimize(f, x0, jac=True, method="L-BFGS-B", bounds=b, options={"maxiter": 2000}).x
    except ImportError:
        return _fit_gd(data, n_views, l2, lr, iters, init)

def _fit_gd(data, n_views, l2=1e-3, lr=0.05, iters=500, init=None):
    """data: list of (X [n_cand, n_views+1], y [n_cand] 0/1). Groups with no positive are skipped."""
    data = [(X, y) for X, y in data if y.sum() > 0 and len(y) > 1]
    theta = np.array(init, float) if init is not None else np.r_[np.full(n_views, 1.0 / max(n_views, 1)), 0.0]
    for _ in range(iters):
        g = l2 * theta
        for X, y in data:
            p = _softmax(X @ theta); t = y / y.sum(); g += X.T @ (p - t) / len(data)
        theta -= lr * g
        theta[:n_views] = np.maximum(theta[:n_views], 0.0)  # keep view weights non-negative
    return theta

def log_loss(data, theta):
    L = [-np.log(max(_softmax(X @ theta)[y > 0].sum(), 1e-12)) for X, y in data if y.sum() > 0]
    return float(np.mean(L)) if L else float("nan")

def ece(confs, correct, bins=10):
    confs, correct = np.asarray(confs, float), np.asarray(correct, float)
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (confs > lo) & (confs <= hi) if lo > 0 else (confs >= lo) & (confs <= hi)
        if m.any(): e += m.mean() * abs(confs[m].mean() - correct[m].mean())
    return float(e)

def probs(X, theta): return _softmax(X @ theta)

def save(theta, views, path):
    json.dump({"views": views, "theta": list(map(float, theta))}, open(path, "w"))
def load(path):
    d = json.load(open(path)); return np.array(d["theta"]), d["views"]


In [ ]:
%%writefile /kaggle/working/select_km1_attempt2.py
# Combined-notebook selection cell (DRAFT, NOT SUBMITTED). Pools ALL candidate pickles from the baseline
# and views4 generation passes, then picks attempt1 = kgmon, attempt2 = calibCV (frozen) or probmul3.
import json, os, numpy as np
from selector.arch_ingest import load_pickles, getter_kgmon, getter_full_probmul_3, to_rows
from selector.calibrate import features, probs
SEL = os.environ.get("ARC_ATTEMPT2", "probmul3")          # "probmul3" (no params) or "calibcv"
V = theta = None
if SEL == "calibcv":                                      # theta only needed for the calibCV variant
    TH = json.load(open(os.environ.get("ARC_THETA", "calibcv_theta_frozen.json")))
    V, theta = TH["views"], np.array(TH["theta"])
def pooled(dirs):
    o = {}
    for d in dirs:
        if not os.path.isdir(d): continue
        for k, s in load_pickles(d).items():
            for h, x in s.items(): o.setdefault(k, {}).setdefault(h, {"grid": x["grid"], "guesses": []})["guesses"] += x["guesses"]
    return o
def safe(f, g):
    try: return f(g)
    except Exception: return -1e9
def pick(sols):
    sols = list(sols.values())
    if not sols: return []
    km = sorted(range(len(sols)), key=lambda i: -safe(getter_kgmon, sols[i]["guesses"]))
    if SEL == "calibcv":
        rows = to_rows({("x", 0): {str(i): s for i, s in enumerate(sols)}}, 8)   # order preserved
        sc = probs(features(rows, V), theta)
    else:
        sc = np.array([safe(getter_full_probmul_3, s["guesses"]) for s in sols])
    a2 = [i for i in np.argsort(-sc, kind="stable") if i != km[0]][:1]
    return [sols[i]["grid"] for i in [km[0]] + a2]
def build(tasks, dirs):
    P = pooled(dirs); sub = {}
    for tid, t in tasks.items():
        sub[tid] = []
        for ti, tt in enumerate(t["test"]):
            g = pick(P.get((tid, ti), {})) or [tt["input"]]
            while len(g) < 2: g.append(g[0])
            sub[tid].append({"attempt_1": g[0], "attempt_2": g[1]})
    return sub


In [ ]:
%%writefile /kaggle/working/f1_confidence.py
# F1 confidence ranking (DRAFT). Reads pass-A pickles, writes task order for pass B (least confident first).
# conf(output) = share of all sampled guesses that hit the kgmon top-1 grid (0 if no candidates).
# conf(task)   = min over its test outputs. Ties broken by task id (deterministic).
import json, os, sys
from selector.arch_ingest import load_pickles, getter_kgmon
def task_conf(dirs, task_ids):
    G = {}
    for d in dirs:
        if os.path.isdir(d):
            for k, s in load_pickles(d).items():
                for h, x in s.items(): G.setdefault(k, {}).setdefault(h, {"grid": x["grid"], "guesses": []})["guesses"] += x["guesses"]
    conf = {t: 1.0 for t in task_ids}; seen = set()
    for (t, i), sols in G.items():
        if t not in conf: continue
        seen.add(t); S = list(sols.values()); n = sum(len(s["guesses"]) for s in S)
        top = max(S, key=lambda s: getter_kgmon(s["guesses"]))
        conf[t] = min(conf[t], len(top["guesses"]) / n if n else 0.0)
    for t in task_ids:
        if t not in seen: conf[t] = 0.0   # pass A produced nothing -> most in need
    return conf
def write_order(dirs, task_ids, path="/kaggle/working/f1_order.json"):
    c = task_conf(dirs, task_ids); order = sorted(task_ids, key=lambda t: (c[t], t))
    json.dump({"order": order, "conf": c}, open(path, "w")); return order, c
if __name__ == "__main__":
    ids = json.load(open(sys.argv[1])).keys(); print(write_order(sys.argv[2:], sorted(ids))[0][:10])


In [ ]:
# 07 — F1 orchestration cell (DRAFT, paste as ONE cell after the %%writefile cells; replaces the single
# `!python starter.py --end-time ...` call). Requires global_end_time from cell 1 (start + 12h - 600 s).
import os, sys, json, time, subprocess, shutil
os.chdir("/kaggle/working")
sys.path.insert(0, "/kaggle/working")
TEST = TEST_CHALLENGES_PATH                         # same path variable the notebook already uses
DIR_A, DIR_B = "/kaggle/working/cands_base", "/kaggle/working/cands_v4"
STOP_B = global_end_time - 20 * 60                  # pass B must be done 20 min before deadline
def run(solver, out_dir, end_t, order=None):
    shutil.copy(solver, "/kaggle/working/arc_solver.py")            # starter.py imports `arc_solver.worker`
    env = dict(os.environ, ARC_OUT_DIR=out_dir, PYTHONHASHSEED="0", UNSLOTH_DISABLE_STATISTICS="1",
               TRITON_PTXAS_PATH="/usr/local/cuda/bin/ptxas", OMP_NUM_THREADS="12")   # as live run cell 8
    env.pop("ARC_TASK_ORDER", None)
    if order: env["ARC_TASK_ORDER"] = order
    for r in range(4):                              # starter's rank handshake files from a prior pass
        try: os.remove(f"/kaggle/worker{r}")
        except FileNotFoundError: pass
    t0 = time.time()
    subprocess.run([sys.executable, "/kaggle/working/starter.py", "--end-time", str(end_t)], env=env, check=False,
                   timeout=max(60, int(end_t - time.time() + 300)))
    print(solver, "took", round((time.time() - t0) / 60, 1), "min; left", round((global_end_time - time.time()) / 60, 1), "min")
# Pass A: full baseline (ARC_GEN_N=2 solver), allowed to run up to STOP_B as well
run("arc_solver_base.py", DIR_A, STOP_B)
# Pass B: views4 on least-confident tasks first, until STOP_B
if STOP_B - time.time() > 1500:                     # need >= one puzzle cap + margin
    from f1_confidence import write_order
    ids = sorted(json.load(open(TEST)).keys())
    if not os.getenv("KAGGLE_IS_COMPETITION_RERUN") and os.environ.get("ARC_SHARD"):
        ids = ids[int(os.environ["ARC_SHARD"])::2]   # smoke: pass B only on the same shard as pass A
    order, conf = write_order([DIR_A], ids, "/kaggle/working/f1_order.json")
    print("pass B order head:", order[:8])
    run("arc_solver_views4.py", DIR_B, STOP_B, "/kaggle/working/f1_order.json")
else:
    print("skip pass B: not enough time")


In [ ]:
# 07 — F1 submission cell (DRAFT). Pools pass A + pass B, attempt_1 = kgmon, attempt_2 = probmul3.
# ARC Prize Kaggle expects /kaggle/working/submission.json ({task: [{"attempt_1":grid,"attempt_2":grid}, ...]}),
# not a CSV. Falls back to pass-A-only kgmon top-2 if anything fails.
import os, sys, json, traceback
sys.path.insert(0, "/kaggle/working")
os.environ.setdefault("ARC_ATTEMPT2", "probmul3")
tasks = json.load(open(TEST_CHALLENGES_PATH))
try:
    from select_km1_attempt2 import build
    sub = build(tasks, ["/kaggle/working/cands_base", "/kaggle/working/cands_v4"])
except Exception:
    traceback.print_exc(); print("F1 SELECT FAILED -> kgmon top-2 on pass A")
    from selector.arch_ingest import load_pickles, baseline_submission
    sub = baseline_submission(load_pickles("/kaggle/working/cands_base"), tasks)
for tid, t in tasks.items():                         # shape guard: one entry per test input
    if len(sub.get(tid, [])) != len(t["test"]):
        sub[tid] = [{"attempt_1": tt["input"], "attempt_2": tt["input"]} for tt in t["test"]]
json.dump(sub, open("/kaggle/working/submission.json", "w")); print("wrote submission.json", len(sub))

# smoke only (non-rerun): score against eval solutions, as the live notebook's reload check did
if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    sol = json.load(open("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json"))
    ids = sorted(tasks)[int(os.environ["ARC_SHARD"])::2] if os.environ.get("ARC_SHARD") else sorted(tasks)
    n = sum(1 for t in ids for i, a in enumerate(sub[t]) if sol[t][i] in (a["attempt_1"], a["attempt_2"]))
    sc = sum(sum(sol[t][i] in (a["attempt_1"], a["attempt_2"]) for i, a in enumerate(sub[t])) / len(sub[t]) for t in ids)
    print(f"*** F1 smoke: {n} correct outputs, task score {sc:.2f}/{len(ids)}")
